In [19]:
import pandas as pd
import numpy as np
import re

In [20]:
df = pd.read_csv('data/raw/WYNA_2497_CTAB_20260424233654.csv', sep=';')

In [21]:
df = df.iloc[:, 1:-1]

df.columns.values[0] = "region"

In [22]:
display(df.head())

,region,ogółem;2014;[zł],ogółem;2015;[zł],ogółem;2016;[zł],ogółem;2017;[zł],ogółem;2018;[zł],ogółem;2019;[zł],ogółem;2020;[zł],ogółem;2021;[zł],ogółem;2022;[zł],ogółem;2023;[zł],ogółem;2024;[zł]
0,DOLNOŚLĄSKIE,"4042,86","4204,24","4385,84","4654,51","4942,39","5323,55","5693,69","6242,88","6945,01","7850,30","8867,46"
1,KUJAWSKO-POMORSKIE,"3439,06","3540,25","3672,98","3886,20","4139,21","4494,37","4831,73","5286,83","5888,55","6721,48","7710,78"
2,LUBELSKIE,"3605,03","3699,48","3815,95","4020,25","4260,71","4564,85","4914,95","5318,68","5909,60","6774,52","7771,05"
3,LUBUSKIE,"3425,38","3567,60","3734,90","3950,95","4239,92","4559,96","4832,07","5286,50","6014,38","6882,46","7887,19"
4,ŁÓDZKIE,"3618,63","3790,76","3925,10","4141,94","4441,29","4790,10","5148,38","5622,29","6210,68","7052,93","8096,32"


In [23]:
print(df.columns)

Index(['region', 'ogółem;2014;[zł]', 'ogółem;2015;[zł]', 'ogółem;2016;[zł]',
       'ogółem;2017;[zł]', 'ogółem;2018;[zł]', 'ogółem;2019;[zł]',
       'ogółem;2020;[zł]', 'ogółem;2021;[zł]', 'ogółem;2022;[zł]',
       'ogółem;2023;[zł]', 'ogółem;2024;[zł]'],
      dtype='object')


In [24]:
import pandas as pd
import re

df_long = df.melt(
    id_vars=["region"],
    var_name="raw_col",
    value_name="wynagrodzenie"
)

def parse(col):
    # jednostka
    unit = re.search(r"\[(.+?)\]", col)
    unit = unit.group(1) if unit else None
    
    # usuń [zł]
    col = re.sub(r"\[.+?\]", "", col)
    
    parts = col.split(";")
    
    # rok = jedyny element 4-cyfrowy
    year = next((p for p in parts if re.fullmatch(r"\d{4}", p)), None)
    
    return pd.Series({
        "rok": int(year) if year else None,
        "jednostka": unit,
        "typ": parts[0]  # tutaj zawsze "ogółem"
    })

df_long = pd.concat(
    [df_long, df_long["raw_col"].apply(parse)],
    axis=1
)

df_long = df_long.drop(columns=["raw_col"])

In [25]:
df_long.drop(columns=["typ", "jednostka"], inplace=True)
display(df_long)

,region,wynagrodzenie,rok
0,DOLNOŚLĄSKIE,"4042,86",2014
1,KUJAWSKO-POMORSKIE,"3439,06",2014
2,LUBELSKIE,"3605,03",2014
3,LUBUSKIE,"3425,38",2014
4,ŁÓDZKIE,"3618,63",2014
...,...,...,...
171,ŚLĄSKIE,"8537,54",2024
172,ŚWIĘTOKRZYSKIE,"7664,18",2024
173,WARMIŃSKO-MAZURSKIE,"7529,41",2024
174,WIELKOPOLSKIE,"7759,22",2024


In [26]:
df_long.to_csv("data/processed/wynagrodzenia.csv", index=False)

In [27]:
df_long.describe()

,rok
count,176.0000
mean,2019.0000
std,3.1713
min,2014.0000
25%,2016.0000
50%,2019.0000
75%,2022.0000
max,2024.0000


In [ ]:
df_long.missing_values = df_long.isnull().sum()
display(df_long.missing_values)

C:\Users\user\AppData\Local\Temp\ipykernel_23324\278907725.py:1: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df_long.missing_values = df_long.isnull().sum()


region           0
wynagrodzenie    0
rok              0
dtype: int64

: 